# V2 — MRI Preprocessing

This notebook preprocesses the selected BraTS 2023 GLI slices for the V2 2D U-Net baseline.

### Pipeline
- 4 MRI modalities: T1n, T1c, T2w, T2f
- Patient-level train/validation split
- Reproducible 1:1 positive/negative slice sampling
- Per-modality Z-score normalization inside the non-zero brain region
- Resize from `240×240` to `128×128`
- Binary tumor mask
- Persistent `.npz` shards for efficient PyTorch loading

**Target:** 16,000 training slices + 4,000 validation slices.


## 1. Configuration and paths


In [1]:
from pathlib import Path
import json
import time
from collections import defaultdict

import nibabel as nib
import numpy as np
from scipy.ndimage import zoom

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = (
    PROJECT_ROOT / "data" / "raw" /
    "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "v2"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "v2"

TARGET_SIZE = 128
SHARD_SIZE = 1000
MODALITIES = ["t1n", "t1c", "t2w", "t2f"]

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET_DIR.exists())
print("Metadata exists:", METADATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)


Project root: /Users/abhra/Downloads/AI-ML/Resume/BraTS
Dataset exists: True
Metadata exists: True
Output directory: /Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v2


## 2. Load reproducible V2 sample metadata

The patient split and sampled slice list were generated previously and saved to disk. This notebook does not resample the dataset.


In [2]:
with open(METADATA_DIR / "patient_split.json") as f:
    patient_split = json.load(f)

with open(METADATA_DIR / "slice_index.json") as f:
    slice_index = json.load(f)

with open(METADATA_DIR / "sampled_slices.json") as f:
    sampled = json.load(f)

print("Training cases:", len(patient_split["train_ids"]))
print("Validation cases:", len(patient_split["val_ids"]))
print("Training samples:", len(sampled["train"]))
print("Validation samples:", len(sampled["val"]))


Training cases: 999
Validation cases: 252
Training samples: 16000
Validation samples: 4000


## 3. Group selected slices by patient

Each patient's four MRI volumes are loaded once, then all selected slices are extracted from that volume.


In [3]:
def group_by_patient(samples):
    grouped = defaultdict(list)
    for patient_id, slice_idx in samples:
        grouped[patient_id].append(int(slice_idx))
    return dict(grouped)

train_patient_slices = group_by_patient(sampled["train"])
val_patient_slices = group_by_patient(sampled["val"])

print("Training patients represented:", len(train_patient_slices))
print("Validation patients represented:", len(val_patient_slices))
print("Average train slices/patient:",
      round(len(sampled["train"]) / len(train_patient_slices), 2))
print("Average val slices/patient:",
      round(len(sampled["val"]) / len(val_patient_slices), 2))


Training patients represented: 999
Validation patients represented: 252
Average train slices/patient: 16.02
Average val slices/patient: 15.87


## 4. Patient preprocessing function

MRI is normalized per modality using non-zero voxels. MRI slices use bilinear interpolation; masks use nearest-neighbor interpolation.


In [4]:
def load_patient(patient_id, selected_slices):
    patient_dir = DATASET_DIR / patient_id

    volumes = []
    for modality in MODALITIES:
        path = patient_dir / f"{patient_id}-{modality}.nii.gz"
        volume = nib.load(path).get_fdata(dtype=np.float32)
        volumes.append(volume)

    mri = np.stack(volumes, axis=0)

    for channel in range(4):
        volume = mri[channel]
        mask = volume > 0
        mean = volume[mask].mean()
        std = volume[mask].std()

        if std > 0:
            volume[mask] = (volume[mask] - mean) / std

        mri[channel] = volume

    seg_path = patient_dir / f"{patient_id}-seg.nii.gz"
    seg = nib.load(seg_path).get_fdata()
    seg = (seg > 0).astype(np.float32)

    selected_slices = np.asarray(selected_slices, dtype=np.int64)

    # [C,H,W,N] -> [N,C,H,W]
    images = np.transpose(
        mri[:, :, :, selected_slices],
        (3, 0, 1, 2)
    )

    # [H,W,N] -> [N,H,W]
    masks = np.transpose(
        seg[:, :, selected_slices],
        (2, 0, 1)
    )

    scale = TARGET_SIZE / 240

    images_resized = zoom(
        images, (1, 1, scale, scale), order=1
    )

    masks_resized = zoom(
        masks, (1, scale, scale), order=0
    )

    return (
        images_resized.astype(np.float32),
        masks_resized.astype(np.float32)
    )


## 5. Benchmark one patient


In [5]:
benchmark_patient = next(iter(train_patient_slices))
benchmark_slices = train_patient_slices[benchmark_patient]

start = time.time()

images, masks = load_patient(
    benchmark_patient,
    benchmark_slices
)

elapsed = time.time() - start

print("Patient:", benchmark_patient)
print("Selected slices:", len(benchmark_slices))
print("Images:", images.shape)
print("Masks:", masks.shape)
print("Images dtype:", images.dtype)
print("Masks dtype:", masks.dtype)
print("Time:", round(elapsed, 2), "seconds")


Patient: BraTS-GLI-01047-000
Selected slices: 16
Images: (16, 4, 128, 128)
Masks: (16, 128, 128)
Images dtype: float32
Masks dtype: float32
Time: 0.36 seconds


## 6. Save processed data as shards

Shards reduce filesystem overhead compared with creating 20,000 individual files. Each shard contains up to 1,000 slices.


In [ ]:
def preprocess_split(samples, split_name):
    output_dir = OUTPUT_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    grouped = group_by_patient(samples)

    images_buffer = []
    masks_buffer = []
    shard_id = 0
    total_samples = 0

    for i, (patient_id, slice_indices) in enumerate(
        grouped.items(), start=1
    ):
        start = time.time()

        images, masks = load_patient(
            patient_id,
            slice_indices
        )

        images_buffer.append(images)
        masks_buffer.append(masks)
        total_samples += len(images)

        current_count = sum(len(x) for x in images_buffer)

        print(
            f"[{i}/{len(grouped)}] {patient_id} | "
            f"{len(slice_indices)} slices | "
            f"{time.time() - start:.2f}s"
        )

        if current_count >= SHARD_SIZE:
            shard_images = np.concatenate(images_buffer, axis=0)
            shard_masks = np.concatenate(masks_buffer, axis=0)

            output_path = output_dir / f"shard_{shard_id:03d}.npz"

            np.savez(
                output_path,
                images=shard_images,
                masks=shard_masks
            )

            print(
                f"  Saved {output_path.name}: "
                f"{len(shard_images)} samples"
            )

            shard_id += 1
            images_buffer = []
            masks_buffer = []

    if images_buffer:
        shard_images = np.concatenate(images_buffer, axis=0)
        shard_masks = np.concatenate(masks_buffer, axis=0)

        output_path = output_dir / f"shard_{shard_id:03d}.npz"

        np.savez(
            output_path,
            images=shard_images,
            masks=shard_masks
        )

        print(
            f"  Saved {output_path.name}: "
            f"{len(shard_images)} samples"
        )

    print(f"\n{split_name}: {total_samples} samples")
    return total_samples


## 7. Process training split

Run this cell to generate the persistent training shards.


In [ ]:
train_total = preprocess_split(
    sampled["train"],
    "train"
)

print("Training preprocessing complete:", train_total)


## 8. Process validation split

Run this cell separately so training and validation preprocessing can be controlled independently.


In [ ]:
val_total = preprocess_split(
    sampled["val"],
    "val"
)

print("Validation preprocessing complete:", val_total)


## 9. Verify processed shards


In [9]:
def inspect_shards(split_name):
    split_dir = OUTPUT_DIR / split_name
    shard_paths = sorted(split_dir.glob("shard_*.npz"))

    total = 0
    print(f"{split_name.upper()} shards: {len(shard_paths)}")

    for path in shard_paths:
        with np.load(path) as data:
            images = data["images"]
            masks = data["masks"]

            print(
                f"{path.name}: "
                f"images={images.shape}, "
                f"masks={masks.shape}"
            )

            total += len(images)

    print("Total samples:", total)
    return total

train_total = inspect_shards("train")
print()
val_total = inspect_shards("val")


TRAIN shards: 16
shard_000.npz: images=(1008, 4, 128, 128), masks=(1008, 128, 128)
shard_001.npz: images=(1010, 4, 128, 128), masks=(1010, 128, 128)
shard_002.npz: images=(1011, 4, 128, 128), masks=(1011, 128, 128)
shard_003.npz: images=(1008, 4, 128, 128), masks=(1008, 128, 128)
shard_004.npz: images=(1008, 4, 128, 128), masks=(1008, 128, 128)
shard_005.npz: images=(1009, 4, 128, 128), masks=(1009, 128, 128)
shard_006.npz: images=(1010, 4, 128, 128), masks=(1010, 128, 128)
shard_007.npz: images=(1010, 4, 128, 128), masks=(1010, 128, 128)
shard_008.npz: images=(1008, 4, 128, 128), masks=(1008, 128, 128)
shard_009.npz: images=(1009, 4, 128, 128), masks=(1009, 128, 128)
shard_010.npz: images=(1009, 4, 128, 128), masks=(1009, 128, 128)
shard_011.npz: images=(1009, 4, 128, 128), masks=(1009, 128, 128)
shard_012.npz: images=(1010, 4, 128, 128), masks=(1010, 128, 128)
shard_013.npz: images=(1009, 4, 128, 128), masks=(1009, 128, 128)
shard_014.npz: images=(1008, 4, 128, 128), masks=(1008, 128

## V2 preprocessing summary

Expected final cache:

| Split | Samples | Image shape |
|---|---:|---|
| Train | 16,000 | `(N, 4, 128, 128)` |
| Validation | 4,000 | `(N, 4, 128, 128)` |

Masks have shape `(N, 128, 128)` and binary values `{0, 1}`.

The resulting shards are consumed by the PyTorch dataset in the next notebook.
